<a href="https://colab.research.google.com/github/NoeliaFerrero/Comi-96160-Data-Science-II-Machine-Learning-para-la-Ciencia-de-Datos-Diplomaturas/blob/main/Semana%205%20Apis%20y%20Data%20Wrangling/API_Ciencia_Datos_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Construimos nuestra propia API de datos

Vamos a crear una API sencilla de ventas con Python + Flask y después la consumiremos como DS.

**Flujo:** Datos → API → JSON → requests → Pandas → merge → análisis


## 🎯 Objetivo

Crear una API educativa con estos endpoints:

- `GET /clientes`
- `GET /productos`
- `GET /ventas`
- `GET /ventas/cliente/<id>`

Después vamos a consumirla con `requests` y convertir las respuestas en DataFrames.


In [1]:
!pip -q install flask
import pandas as pd
import requests
from flask import Flask, jsonify


# 1. Creamos los datos

Imaginemos una empresa que quiere exponer sus datos de clientes, productos y ventas para que otras aplicaciones puedan consultarlos.


In [2]:
clientes = [
    {'id_cliente': 1, 'nombre': 'Ana', 'ciudad': 'Cordoba'},
    {'id_cliente': 2, 'nombre': 'Bruno', 'ciudad': 'Rosario'},
    {'id_cliente': 3, 'nombre': 'Carla', 'ciudad': 'Cordoba'},
    {'id_cliente': 4, 'nombre': 'Diego', 'ciudad': 'Mendoza'},
    {'id_cliente': 5, 'nombre': 'Elena', 'ciudad': 'Cordoba'}
]

productos = [
    {'id_producto': 101, 'producto': 'Notebook', 'categoria': 'Tecnologia', 'precio': 850000},
    {'id_producto': 102, 'producto': 'Monitor', 'categoria': 'Tecnologia', 'precio': 320000},
    {'id_producto': 103, 'producto': 'Silla', 'categoria': 'Muebles', 'precio': 180000},
    {'id_producto': 104, 'producto': 'Escritorio', 'categoria': 'Muebles', 'precio': 250000}
]

ventas = [
    {'id_venta': 1, 'id_cliente': 1, 'id_producto': 101, 'cantidad': 1},
    {'id_venta': 2, 'id_cliente': 1, 'id_producto': 102, 'cantidad': 2},
    {'id_venta': 3, 'id_cliente': 2, 'id_producto': 103, 'cantidad': 3},
    {'id_venta': 4, 'id_cliente': 3, 'id_producto': 101, 'cantidad': 1},
    {'id_venta': 5, 'id_cliente': 3, 'id_producto': 104, 'cantidad': 2},
    {'id_venta': 6, 'id_cliente': 4, 'id_producto': 102, 'cantidad': 1},
    {'id_venta': 7, 'id_cliente': 5, 'id_producto': 103, 'cantidad': 2}
]

print(len(clientes), 'clientes')
print(len(productos), 'productos')
print(len(ventas), 'ventas')


5 clientes
4 productos
7 ventas


# 2. Construimos la API 🚪

Una API tiene **endpoints**, que son puertas de acceso a recursos.

Por ejemplo: `GET /clientes` significa 'quiero consultar clientes'.


In [3]:
app = Flask(__name__)

@app.route('/clientes', methods=['GET'])
def obtener_clientes():
    return jsonify(clientes)

@app.route('/productos', methods=['GET'])
def obtener_productos():
    return jsonify(productos)

@app.route('/ventas', methods=['GET'])
def obtener_ventas():
    return jsonify(ventas)

@app.route('/ventas/cliente/<int:id_cliente>', methods=['GET'])
def ventas_por_cliente(id_cliente):
    resultado = [v for v in ventas if v['id_cliente'] == id_cliente]
    return jsonify(resultado)

print('API creada 🚀')


API creada 🚀


# 3. Encendemos la API en Colab

La API correrá dentro del entorno de Colab. Para la clase esto alcanza: no necesitamos publicar nada en Internet.


In [4]:
import threading, time

def ejecutar_api():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

threading.Thread(target=ejecutar_api, daemon=True).start()
time.sleep(2)
print('API disponible en http://127.0.0.1:5000')


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


API disponible en http://127.0.0.1:5000


# 4. La consumimos como DS 📥

Ahora hacemos una solicitud HTTP a nuestra propia API.


In [5]:
respuesta = requests.get('http://127.0.0.1:5000/clientes')

print('Status code:', respuesta.status_code)
print('Respuesta JSON:')
respuesta.json()


INFO:werkzeug:127.0.0.1 - - [10/Aug/2026 22:39:07] "GET /clientes HTTP/1.1" 200 -


Status code: 200
Respuesta JSON:


[{'ciudad': 'Cordoba', 'id_cliente': 1, 'nombre': 'Ana'},
 {'ciudad': 'Rosario', 'id_cliente': 2, 'nombre': 'Bruno'},
 {'ciudad': 'Cordoba', 'id_cliente': 3, 'nombre': 'Carla'},
 {'ciudad': 'Mendoza', 'id_cliente': 4, 'nombre': 'Diego'},
 {'ciudad': 'Cordoba', 'id_cliente': 5, 'nombre': 'Elena'}]

## 🧠 ¿Qué pasó?

`requests.get()` hizo una petición a un endpoint.

La API respondió JSON.

Es decir:

**Python → API → JSON → Python**


In [6]:
df_clientes = pd.DataFrame(respuesta.json())
df_clientes


,ciudad,id_cliente,nombre
0,Cordoba,1,Ana
1,Rosario,2,Bruno
2,Cordoba,3,Carla
3,Mendoza,4,Diego
4,Cordoba,5,Elena


# 5. Consultamos productos y ventas


In [7]:
df_productos = pd.DataFrame(
    requests.get('http://127.0.0.1:5000/productos').json()
)

df_ventas = pd.DataFrame(
    requests.get('http://127.0.0.1:5000/ventas').json()
)

display(df_productos)
display(df_ventas)


INFO:werkzeug:127.0.0.1 - - [10/Aug/2026 22:39:28] "GET /productos HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Aug/2026 22:39:28] "GET /ventas HTTP/1.1" 200 -


,categoria,id_producto,precio,producto
0,Tecnologia,101,850000,Notebook
1,Tecnologia,102,320000,Monitor
2,Muebles,103,180000,Silla
3,Muebles,104,250000,Escritorio


,cantidad,id_cliente,id_producto,id_venta
0,1,1,101,1
1,2,1,102,2
2,3,2,103,3
3,1,3,101,4
4,2,3,104,5
5,1,4,102,6
6,2,5,103,7


# 6. Data Wrangling + merge 🔗

Tenemos tres fuentes. Ahora las relacionamos usando sus claves.


In [8]:
df = df_ventas.merge(df_clientes, on='id_cliente', how='left')
df = df.merge(df_productos, on='id_producto', how='left')

df['importe'] = df['cantidad'] * df['precio']
df


,cantidad,id_cliente,id_producto,id_venta,ciudad,nombre,categoria,precio,producto,importe
0,1,1,101,1,Cordoba,Ana,Tecnologia,850000,Notebook,850000
1,2,1,102,2,Cordoba,Ana,Tecnologia,320000,Monitor,640000
2,3,2,103,3,Rosario,Bruno,Muebles,180000,Silla,540000
3,1,3,101,4,Cordoba,Carla,Tecnologia,850000,Notebook,850000
4,2,3,104,5,Cordoba,Carla,Muebles,250000,Escritorio,500000
5,1,4,102,6,Mendoza,Diego,Tecnologia,320000,Monitor,320000
6,2,5,103,7,Cordoba,Elena,Muebles,180000,Silla,360000


# 7. Respondemos preguntas de negocio 📊

¿Cuánto vendió cada cliente? ¿Qué ciudad factura más? ¿Qué producto genera más ingresos?


In [9]:
ventas_cliente = (df.groupby(['id_cliente','nombre','ciudad'], as_index=False)['importe']
                   .sum().sort_values('importe', ascending=False))
display(ventas_cliente)

ventas_ciudad = (df.groupby('ciudad', as_index=False)['importe']
                  .sum().sort_values('importe', ascending=False))
display(ventas_ciudad)

ventas_producto = (df.groupby('producto', as_index=False)['importe']
                    .sum().sort_values('importe', ascending=False))
display(ventas_producto)


,id_cliente,nombre,ciudad,importe
0,1,Ana,Cordoba,1490000
2,3,Carla,Cordoba,1350000
1,2,Bruno,Rosario,540000
4,5,Elena,Cordoba,360000
3,4,Diego,Mendoza,320000


,ciudad,importe
0,Cordoba,3200000
2,Rosario,540000
1,Mendoza,320000


,producto,importe
2,Notebook,1700000
1,Monitor,960000
3,Silla,900000
0,Escritorio,500000


# 8. Endpoint con filtro 🎯

También podemos pedir solo las ventas de un cliente.

Ejemplo: `GET /ventas/cliente/1`


In [10]:
id_buscado = 1
url = f'http://127.0.0.1:5000/ventas/cliente/{id_buscado}'
r = requests.get(url)
print('Status:', r.status_code)
pd.DataFrame(r.json())


INFO:werkzeug:127.0.0.1 - - [10/Aug/2026 22:40:03] "GET /ventas/cliente/1 HTTP/1.1" 200 -


Status: 200


,cantidad,id_cliente,id_producto,id_venta
0,1,1,101,1
1,2,1,102,2


# 🕵️ DESAFÍO FINAL

Sin mirar la solución, respondé usando la API y Pandas:

1. ¿Qué producto genera mayor facturación?
2. ¿Qué cliente gastó más?
3. ¿Qué ciudad genera más ingresos?
4. ¿Qué cliente compró más unidades?
5. ⭐ Bonus: crear `resumen_cliente(id_cliente)` que devuelva nombre, ciudad, cantidad de compras, unidades y facturación total.

### Pistas
Usá `requests`, `pd.DataFrame()`, `merge()` y `groupby()`.


In [11]:
# SOLUCIÓN

print('Producto con mayor facturación:')
display(ventas_producto.head(1))

print('Cliente que más gastó:')
display(ventas_cliente.head(1))

print('Ciudad con mayor facturación:')
display(ventas_ciudad.head(1))

unidades = (df.groupby(['id_cliente','nombre'], as_index=False)['cantidad']
            .sum().sort_values('cantidad', ascending=False))
print('Cliente con más unidades:')
display(unidades.head(1))

def resumen_cliente(id_cliente):
    d = df[df['id_cliente'] == id_cliente]
    if d.empty:
        return 'Cliente no encontrado'
    return {
        'nombre': d['nombre'].iloc[0],
        'ciudad': d['ciudad'].iloc[0],
        'cantidad_compras': d['id_venta'].nunique(),
        'unidades_compradas': d['cantidad'].sum(),
        'facturacion_total': d['importe'].sum()
    }

resumen_cliente(1)


Producto con mayor facturación:


,producto,importe
2,Notebook,1700000


Cliente que más gastó:


,id_cliente,nombre,ciudad,importe
0,1,Ana,Cordoba,1490000


Ciudad con mayor facturación:


,ciudad,importe
0,Cordoba,3200000


Cliente con más unidades:


,id_cliente,nombre,cantidad
0,1,Ana,3


{'nombre': 'Ana',
 'ciudad': 'Cordoba',
 'cantidad_compras': 2,
 'unidades_compradas': np.int64(3),
 'facturacion_total': np.int64(1490000)}

# 🎉 Cierre

Hoy hicimos un mini pipeline realista:

`DATOS → API → ENDPOINT → JSON → requests → DataFrame → Wrangling → merge → análisis`

> Una API puede ser una **fuente de datos** para un proyecto de DS.
